In [50]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# siffatkhan_eeg_dataset1_path = kagglehub.dataset_download('siffatkhan/eeg-dataset1')

# print('Data source import complete.')


In [51]:
"""
=============================================================
  Epileptic Seizure Dataset — Preprocessing Pipelines
  Step 2: Pipeline A & Pipeline B (Standalone, Runnable)
=============================================================

Dataset: UCI Epileptic Seizure Recognition
  - 11,500 samples × 178 EEG amplitude features
  - Binary label: 1 = seizure, 0 = non-seizure (1:4 imbalance)


"""

import argparse
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD / SIMULATE DATA
# ─────────────────────────────────────────────────────────────────────────────

def load_data(csv_path=None):
    """
    If a real CSV path is given, load it and binarise the label
    (Class 1 = seizure → 1; Classes 2-5 → 0).
    Otherwise simulate a dataset that mirrors UCI properties.
    """
    if csv_path:
        print(f"[Data] Loading real dataset from: {csv_path}")
        df = pd.read_csv(csv_path)
        # UCI column layout: X1 … X178, y (1-5)
        feature_cols = [c for c in df.columns if c.startswith('X')]
        X = df[feature_cols].values.astype(float)
        y = (df['y'].values == 1).astype(int)   # 1 = seizure, 0 = non-seizure
    else:
        print("[Data] Simulating UCI-like dataset (11 500 samples, 178 features, 1:4 imbalance)")
        X, y = make_classification(
            n_samples=11_500,
            n_features=178,
            n_informative=40,
            n_redundant=30,
            n_clusters_per_class=3,
            weights=[0.80, 0.20],
            flip_y=0.03,
            random_state=42,
        )
        X += np.random.RandomState(42).normal(0, 0.5, X.shape)   # mild EEG-like noise

    print(f"   Shape  : {X.shape}")
    print(f"   Classes: Seizure={y.sum():,}  Non-seizure={(1-y).sum():,}  "
          f"Ratio={((1-y).sum()/y.sum()):.1f}:1\n")
    return X, y




In [52]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────────────────────────

def split_data(X, y, test_size=0.20, val_size=0.20, random_state=42):
    """
    Returns:
        X_train, X_val, X_test   (feature arrays)
        y_train, y_val, y_test   (label arrays)

    Split strategy
    ──────────────
    Full data  ──80/20──►  Train pool | Test
    Train pool ──80/20──►  Train      | Validation
    Stratification preserves the 1:4 class ratio in every split.
    """
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_pool, y_train_pool,
        test_size=val_size, stratify=y_train_pool, random_state=random_state)

    print("[Split]")
    print(f"   Train : {X_train.shape}  "
          f"(seizure={y_train.sum()}, non={( 1-y_train).sum()})")
    print(f"   Val   : {X_val.shape}  "
          f"(seizure={y_val.sum()}, non={(1-y_val).sum()})")
    print(f"   Test  : {X_test.shape}  "
          f"(seizure={y_test.sum()}, non={(1-y_test).sum()})\n")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [53]:
# ─────────────────────────────────────────────────────────────────────────────
# 2-A.  PIPELINE A
#       MinMax Normalisation → SelectKBest (ANOVA F-test) → StandardScaler
# ─────────────────────────────────────────────────────────────────────────────

def build_pipeline_A(k=60):
    """
    Pipeline A: Normalisation → Noise removal / Feature selection → Re-scaling

    Step 1 — MinMaxScaler
        Squeezes every feature into [0, 1].
        Removes absolute amplitude differences caused by electrode placement,
        skin resistance, or device gain — common in raw EEG recordings.

    Step 2 — SelectKBest (f_classif, k=60)
        ANOVA F-test scores each of the 178 time-point columns by how well
        they separate seizure from non-seizure classes.
        The top-60 most discriminative time points are retained.

        Research insight:  Selecting features *before* re-scaling ensures
        the subsequent StandardScaler only normalises genuinely informative
        columns, avoiding 'wasting' standardisation budget on noise.
        Empirically, early feature selection also speeds up LR convergence.

    Step 3 — StandardScaler
        Zero-mean, unit-variance standardisation on the selected 60 features.
        Required by Logistic Regression (L2 regularisation penalises raw
        magnitudes; un-scaled features bias the penalty unfairly).

    Parameters
    ----------
    k : int
        Number of top features to keep (default 60, ~34 % of 178).
    """
    pipe = Pipeline([
        ('step1_minmax',   MinMaxScaler()),
        ('step2_kbest',    SelectKBest(score_func=f_classif, k=k)),
        ('step3_standard', StandardScaler()),
    ])
    return pipe


def run_pipeline_A(X_train, X_val, X_test, y_train, k=60, verbose=True):
    """
    Fit Pipeline A on training data; transform val & test without data leakage.

    Returns
    -------
    pipe_A   : fitted sklearn Pipeline
    X_tr_A   : transformed training features  (n_train, k)
    X_val_A  : transformed validation features (n_val,  k)
    X_test_A : transformed test features       (n_test, k)
    selected_indices : original column indices of retained features
    f_scores : ANOVA F-scores for all 178 features (before selection)
    """
    pipe_A = build_pipeline_A(k=k)

    # fit_transform on train only — prevents leakage into val/test
    X_tr_A   = pipe_A.fit_transform(X_train, y_train)
    X_val_A  = pipe_A.transform(X_val)
    X_test_A = pipe_A.transform(X_test)

    # Diagnostic info
    selector        = pipe_A.named_steps['step2_kbest']
    selected_mask   = selector.get_support()
    selected_indices = np.where(selected_mask)[0]
    f_scores         = selector.scores_             # scores for all 178 cols
    f_pvalues        = selector.pvalues_

    if verbose:
        print("=" * 55)
        print("  PIPELINE A RESULTS")
        print("=" * 55)
        print(f"  Input features  : {X_train.shape[1]}")
        print(f"  Selected (k)    : {k}")
        print(f"  Train shape     : {X_tr_A.shape}")
        print(f"  Val   shape     : {X_val_A.shape}")
        print(f"  Test  shape     : {X_test_A.shape}")
        print(f"\n  Top-10 selected feature indices (time-point columns):")
        top10_local  = np.argsort(f_scores[selected_mask])[::-1][:10]
        top10_global = selected_indices[top10_local]
        for rank, (g, f) in enumerate(
                zip(top10_global, f_scores[selected_mask][top10_local]), 1):
            print(f"    {rank:2d}. X{g+1:<4d}  F-score = {f:.2f}")
        print()

    return pipe_A, X_tr_A, X_val_A, X_test_A, selected_indices, f_scores


In [54]:
# ─────────────────────────────────────────────────────────────────────────────
# 2-B.  PIPELINE B
#       StandardScaler → PCA (95 % variance) → MinMaxScaler
# ─────────────────────────────────────────────────────────────────────────────

def build_pipeline_B(variance_threshold=0.95):
    """
    Pipeline B: Feature extraction (PCA) → Scaling → Re-scaling

    Step 1 — StandardScaler
        Zero-mean, unit-variance before PCA.
        PCA is variance-maximising: without standardisation, high-amplitude
        columns dominate the principal components regardless of class relevance.
        Mandatory pre-step for correct PCA behaviour on EEG data.

    Step 2 — PCA (n_components = 0.95)
        Finds the linear combinations of the 178 time points that capture
        95 % of total signal variance.

        Research insight:  Adjacent EEG time points within a 1-second window
        are highly correlated (autocorrelation of neural oscillations).
        PCA exploits this redundancy — typically reducing 178 features to
        ~100-140 orthogonal components while retaining almost all information.
        Orthogonal components also remove multicollinearity, which can inflate
        LR coefficient variance.

    Step 3 — MinMaxScaler
        Re-scales PCA scores to [0, 1].
        PCA components have varying standard deviations (eigenvalue-scaled);
        MinMax brings them to a common range before passing to the classifier.

    Parameters
    ----------
    variance_threshold : float  (default 0.95)
        Fraction of variance to retain.  Tune between 0.90 and 0.99.
    """
    pipe = Pipeline([
        ('step1_standard', StandardScaler()),
        ('step2_pca',      PCA(n_components=variance_threshold, random_state=42)),
        ('step3_minmax',   MinMaxScaler()),
    ])
    return pipe


def run_pipeline_B(X_train, X_val, X_test, y_train,
                   variance_threshold=0.95, verbose=True):
    """
    Fit Pipeline B on training data; transform val & test without data leakage.

    Returns
    -------
    pipe_B            : fitted sklearn Pipeline
    X_tr_B            : transformed training features  (n_train, n_pca)
    X_val_B           : transformed validation features (n_val,  n_pca)
    X_test_B          : transformed test features       (n_test, n_pca)
    explained_var     : cumulative explained variance ratio array
    n_components      : number of PCA components retained
    """
    pipe_B = build_pipeline_B(variance_threshold=variance_threshold)

    X_tr_B   = pipe_B.fit_transform(X_train, y_train)
    X_val_B  = pipe_B.transform(X_val)
    X_test_B = pipe_B.transform(X_test)

    pca           = pipe_B.named_steps['step2_pca']
    n_components  = pca.n_components_
    explained_var = np.cumsum(pca.explained_variance_ratio_)

    if verbose:
        print("=" * 55)
        print("  PIPELINE B RESULTS")
        print("=" * 55)
        print(f"  Input features          : {X_train.shape[1]}")
        print(f"  Variance threshold      : {variance_threshold*100:.0f}%")
        print(f"  PCA components retained : {n_components}")
        print(f"  Compression ratio       : {X_train.shape[1]/n_components:.2f}x")
        print(f"  Train shape             : {X_tr_B.shape}")
        print(f"  Val   shape             : {X_val_B.shape}")
        print(f"  Test  shape             : {X_test_B.shape}")
        print(f"\n  Variance explained by component:")
        for i in [1, 5, 10, 20, 50, n_components]:
            i = min(i, n_components)
            print(f"    First {i:3d} components → {explained_var[i-1]*100:.2f}% variance")
        print()

    return pipe_B, X_tr_B, X_val_B, X_test_B, explained_var, n_components




In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  VISUALISATION
# ─────────────────────────────────────────────────────────────────────────────

def plot_pipeline_A_analysis(f_scores, selected_indices, X_tr_A, y_train, k):
    """Three-panel analysis plot for Pipeline A."""
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Pipeline A — MinMax → SelectKBest → StandardScaler",
                 fontsize=14, fontweight='bold')

    # Panel 1: F-score bar for all 178 features, highlight selected
    ax = axes[0]
    colors = ['#E63946' if i in selected_indices else '#A8DADC'
              for i in range(len(f_scores))]
    ax.bar(range(len(f_scores)), f_scores, color=colors, width=1.0, edgecolor='none')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#E63946', label=f'Selected (top {k})'),
                        Patch(color='#A8DADC', label='Rejected')],
              fontsize=9, framealpha=0.4)
    ax.set_xlabel('Feature Index (X1 – X178)', fontsize=10)
    ax.set_ylabel('ANOVA F-Score', fontsize=10)
    ax.set_title('Feature F-Scores\n(ANOVA per time-point column)', fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 2: Distribution of selected vs rejected F-scores
    ax = axes[1]
    mask_sel = np.zeros(len(f_scores), dtype=bool)
    mask_sel[selected_indices] = True
    ax.hist(f_scores[~mask_sel], bins=30, color='#A8DADC', alpha=0.7,
            label='Rejected', density=True)
    ax.hist(f_scores[mask_sel],  bins=20, color='#E63946', alpha=0.7,
            label=f'Selected (top {k})', density=True)
    ax.set_xlabel('F-Score', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title('F-Score Distribution\nSelected vs Rejected', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 3: Mean EEG amplitude profile per class after Pipeline A
    ax = axes[2]
    mean_seizure    = X_tr_A[y_train == 1].mean(axis=0)
    mean_nonseizure = X_tr_A[y_train == 0].mean(axis=0)
    ax.plot(mean_seizure,    color='#E63946', lw=1.8, label='Seizure (class 1)',    alpha=0.9)
    ax.plot(mean_nonseizure, color='#457B9D', lw=1.8, label='Non-seizure (class 0)', alpha=0.9)
    ax.fill_between(range(k), mean_seizure, mean_nonseizure, alpha=0.10, color='#E63946')
    ax.set_xlabel(f'Selected Feature Index (0–{k-1})', fontsize=10)
    ax.set_ylabel('Std-Scaled Amplitude (mean)', fontsize=10)
    ax.set_title('Mean Feature Profile\nSeizure vs Non-Seizure (after Pipe A)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    # out = '/mnt/user-data/outputs/pipeline_A_analysis.png'
    # out = '/kaggle/working/pipeline_A_analysis.png'
    out = '/content/pipeline_A_analysis.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    # plt.close()
    print(f"  [Saved] {out}")


def plot_pipeline_B_analysis(pipe_B, explained_var, n_components, X_tr_B, y_train):
    """Three-panel analysis plot for Pipeline B."""
    pca = pipe_B.named_steps['step2_pca']
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("Pipeline B — StandardScaler → PCA → MinMax",
                 fontsize=14, fontweight='bold')

    # Panel 1: Cumulative explained variance
    ax = axes[0]
    ax.plot(range(1, n_components + 1), explained_var * 100,
            color='#2A9D8F', lw=2.5)
    ax.axhline(95, color='#E9C46A', ls='--', lw=1.8, label='95% threshold')
    ax.axvline(n_components, color='#E63946', ls=':', lw=1.8,
               label=f'n={n_components} components')
    ax.fill_between(range(1, n_components + 1), explained_var * 100,
                    alpha=0.12, color='#2A9D8F')
    ax.set_xlabel('Number of PCA Components', fontsize=10)
    ax.set_ylabel('Cumulative Explained Variance (%)', fontsize=10)
    ax.set_title('PCA Scree Plot\n(Cumulative Variance)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 2: Individual explained variance per component (first 40)
    ax = axes[1]
    evr = pca.explained_variance_ratio_[:40] * 100
    ax.bar(range(1, len(evr) + 1), evr, color='#2A9D8F', edgecolor='none')
    ax.set_xlabel('PCA Component', fontsize=10)
    ax.set_ylabel('Explained Variance (%)', fontsize=10)
    ax.set_title('Per-Component Variance\n(First 40 components)', fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 3: Scatter of PC1 vs PC2 coloured by class
    ax = axes[2]
    idx0 = np.where(y_train == 0)[0][:800]
    idx1 = np.where(y_train == 1)[0][:800]
    ax.scatter(X_tr_B[idx0, 0], X_tr_B[idx0, 1],
               c='#457B9D', alpha=0.35, s=8, label='Non-seizure')
    ax.scatter(X_tr_B[idx1, 0], X_tr_B[idx1, 1],
               c='#E63946', alpha=0.55, s=8, label='Seizure')
    ax.set_xlabel('PC 1 (MinMax-scaled)', fontsize=10)
    ax.set_ylabel('PC 2 (MinMax-scaled)', fontsize=10)
    ax.set_title('PC1 vs PC2 Projection\n(800 samples/class)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    # out = '/mnt/user-data/outputs/pipeline_B_analysis.png'
    # Pipeline B
    # out = '/kaggle/working/pipeline_B_analysis.png'
    out = '/content/pipeline_B_analysis.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    # plt.close()
    print(f"  [Saved] {out}")


def plot_pipeline_comparison(X_train, X_tr_A, X_tr_B, n_components, k):
    """Side-by-side comparison of both pipelines."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Pipeline A vs Pipeline B — Head-to-Head Comparison",
                 fontsize=14, fontweight='bold')

    # Panel 1: Dimensionality reduction
    ax = axes[0]
    labels = [f'Raw\n({X_train.shape[1]} feat)',
              f'Pipeline A\n({k} feat)',
              f'Pipeline B\n(PCA {n_components})']
    vals   = [X_train.shape[1], k, n_components]
    cols   = ['#264653', '#2A9D8F', '#E9C46A']
    bars   = ax.bar(labels, vals, color=cols, edgecolor='white', width=0.45)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(v), ha='center', va='bottom', fontsize=13, fontweight='bold')
    ax.set_ylabel('Dimensions', fontsize=10)
    ax.set_title('Dimensionality After\nEach Pipeline', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 210)
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 2: Feature range after each pipeline (box)
    ax = axes[1]
    sample_idx = np.random.choice(X_train.shape[0], 500, replace=False)
    data_raw = X_train[sample_idx, :10].flatten()
    data_A   = X_tr_A[sample_idx, :10].flatten()
    data_B   = X_tr_B[sample_idx, :10].flatten()
    bp = ax.boxplot([data_raw, data_A, data_B],
                    tick_labels=['Raw', 'Pipeline A', 'Pipeline B'],
                    patch_artist=True, widths=0.4,
                    medianprops=dict(color='white', lw=2))
    for patch, col in zip(bp['boxes'], ['#264653', '#2A9D8F', '#E9C46A']):
        patch.set_facecolor(col)
        patch.set_alpha(0.85)
    ax.set_ylabel('Feature Value Distribution', fontsize=10)
    ax.set_title('Value Range Comparison\n(first 10 features, 500 samples)', fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

    # Panel 3: Text summary card
    ax = axes[2]
    ax.axis('off')
    summary = (
        "PIPELINE DESIGN DECISIONS\n"
        "─────────────────────────────────────\n\n"
        "PIPELINE A\n"
        "  • MinMax first: removes amplitude\n"
        "    bias from electrode placement\n"
        "  • SelectKBest: retains only the 60\n"
        "    most class-discriminative time\n"
        "    points (ANOVA F-test, p<0.05)\n"
        "  • StandardScaler last: prepares\n"
        "    features for L2-regularised LR\n\n"
        "  ✔ Best for: tabular EEG with known\n"
        "    discriminative time windows\n\n"
        "─────────────────────────────────────\n\n"
        "PIPELINE B\n"
        "  • StandardScaler first: mandatory\n"
        "    before PCA (variance equalisation)\n"
        f"  • PCA: {n_components} orthogonal components\n"
        "    capture 95% of EEG variance\n"
        "  • MinMax last: normalises PCA\n"
        "    scores to common [0,1] range\n\n"
        "  ✔ Best for: correlated EEG signals\n"
        "    where temporal autocorrelation\n"
        "    creates redundant features\n\n"
        "─────────────────────────────────────\n"
        "KEY INSIGHT: Ordering matters.\n"
        "  SelectKBest before scaling (A)\n"
        "  prevents noise features from\n"
        "  consuming regularisation budget."
    )
    ax.text(0.05, 0.97, summary, transform=ax.transAxes,
            va='top', ha='left', fontsize=8.5,
            fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f0f4f8',
                      edgecolor='#b2bec3', alpha=0.9))

    plt.tight_layout()
    # out = '/mnt/user-data/outputs/pipeline_comparison.png'
    # out = '/kaggle/working/pipeline_comparison.png'
    out = '/content/pipeline_comparison.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    plt.show()
    # plt.close()
    print(f"  [Saved] {out}")



In [56]:

# ─────────────────────────────────────────────────────────────────────────────
# 4.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description='Epileptic Seizure Preprocessing Pipelines')
    parser.add_argument('--csv', type=str, default=None,
                        help='Path to "Epileptic Seizure Recognition.csv" (optional)')
    parser.add_argument('--k',   type=int, default=60,
                        help='SelectKBest k for Pipeline A (default: 60)')
    parser.add_argument('--var', type=float, default=0.95,
                        help='PCA variance threshold for Pipeline B (default: 0.95)')
    args = parser.parse_args(args=[])

    np.random.seed(42)

    print("\n" + "=" * 55)
    print("  EPILEPTIC SEIZURE — PREPROCESSING PIPELINES")
    print("=" * 55 + "\n")

    # ── Load data ──────────────────────────────────────────
    X, y = load_data(csv_path='/content/Epileptic Seizure Recognition.csv')

    # ── Split ──────────────────────────────────────────────
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    # ── Pipeline A ─────────────────────────────────────────
    print("=" * 55)
    print("  RUNNING PIPELINE A")
    print("=" * 55)
    pipe_A, X_tr_A, X_val_A, X_test_A, sel_idx, f_scores = run_pipeline_A(
        X_train, X_val, X_test, y_train, k=args.k)

    # ── Pipeline B ─────────────────────────────────────────
    print("=" * 55)
    print("  RUNNING PIPELINE B")
    print("=" * 55)
    pipe_B, X_tr_B, X_val_B, X_test_B, expl_var, n_comp = run_pipeline_B(
        X_train, X_val, X_test, y_train, variance_threshold=args.var)

    # ── Plots ──────────────────────────────────────────────
    print("=" * 55)
    print("  GENERATING PLOTS")
    print("=" * 55)
    plot_pipeline_A_analysis(f_scores, sel_idx, X_tr_A, y_train, args.k)
    plot_pipeline_B_analysis(pipe_B, expl_var, n_comp, X_tr_B, y_train)
    plot_pipeline_comparison(X_train, X_tr_A, X_tr_B, n_comp, args.k)

    # ── Verify no data leakage ─────────────────────────────
    print("\n" + "=" * 55)
    print("  DATA LEAKAGE CHECK")
    print("=" * 55)
    print("  Scalers/selectors fitted on TRAINING set only:")
    for name, step in pipe_A.steps:
        fitted = hasattr(step, 'n_features_in_') or hasattr(step, 'scale_')
        print(f"    Pipeline A / {name:<18s}: fitted={fitted}")
    for name, step in pipe_B.steps:
        fitted = hasattr(step, 'n_features_in_') or hasattr(step, 'scale_')
        print(f"    Pipeline B / {name:<18s}: fitted={fitted}")

    # ── Return all arrays for downstream use ───────────────
    print("\n" + "=" * 55)
    print("  OUTPUTS READY FOR STEP 3 (LOGISTIC REGRESSION)")
    print("=" * 55)
    print("  Pipeline A arrays:  X_tr_A, X_val_A, X_test_A")
    print("  Pipeline B arrays:  X_tr_B, X_val_B, X_test_B")
    print("  Labels:             y_train, y_val, y_test")
    print("\n  Pass these directly into LogisticRegression().fit(X_tr_A, y_train)")
    print()

    return dict(
        pipe_A=pipe_A, pipe_B=pipe_B,
        X_tr_A=X_tr_A, X_val_A=X_val_A, X_test_A=X_test_A,
        X_tr_B=X_tr_B, X_val_B=X_val_B, X_test_B=X_test_B,
        y_train=y_train, y_val=y_val, y_test=y_test,
    )


if __name__ == '__main__':
    main()


  EPILEPTIC SEIZURE — PREPROCESSING PIPELINES

[Data] Loading real dataset from: /content/Epileptic Seizure Recognition.csv
   Shape  : (11500, 178)
   Classes: Seizure=2,300  Non-seizure=9,200  Ratio=4.0:1

[Split]
   Train : (7360, 178)  (seizure=1472, non=5888)
   Val   : (1840, 178)  (seizure=368, non=1472)
   Test  : (2300, 178)  (seizure=460, non=1840)

  RUNNING PIPELINE A
  PIPELINE A RESULTS
  Input features  : 178
  Selected (k)    : 60
  Train shape     : (7360, 60)
  Val   shape     : (1840, 60)
  Test  shape     : (2300, 60)

  Top-10 selected feature indices (time-point columns):
     1. X10    F-score = 45.87
     2. X7     F-score = 45.70
     3. X8     F-score = 45.60
     4. X9     F-score = 43.83
     5. X11    F-score = 43.27
     6. X12    F-score = 35.95
     7. X6     F-score = 33.95
     8. X35    F-score = 23.86
     9. X158   F-score = 22.16
    10. X178   F-score = 22.13

  RUNNING PIPELINE B
  PIPELINE B RESULTS
  Input features          : 178
  Variance th

In [57]:
# ============================================================
# BLOCK 1 — IMPORTS & GLOBAL SETTINGS
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

from sklearn.datasets import make_classification

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    learning_curve,
    validation_curve,
    cross_val_score
)

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler
)

from sklearn.feature_selection import (
    SelectKBest,
    f_classif
)

from sklearn.decomposition import PCA

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUT = "/mnt/user-data/outputs"

# ============================================================
# COLOR PALETTE
# ============================================================

C_BLUE    = "#2E75B6"
C_ORANGE  = "#E76F51"
C_GREEN   = "#2A9D8F"
C_RED     = "#E63946"
C_PURPLE  = "#7B2D8B"
C_GREY    = "#6C757D"
C_TEAL    = "#0077B6"

BG = "#F7F9FC"

# ============================================================
# MATPLOTLIB STYLING
# ============================================================

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.facecolor'   : BG,
    'figure.facecolor' : 'white',
    'axes.titleweight' : 'bold',
    'axes.titlesize'   : 12,
    'axes.labelsize'   : 10,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
    'legend.framealpha': 0.5,
})

In [58]:
# ============================================================
# BLOCK 2 — DATASET GENERATION & SPLIT
# ============================================================

print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

np.random.seed(42)

X_raw, y = make_classification(
    n_samples=11500,
    n_features=178,
    n_informative=40,
    n_redundant=30,
    n_clusters_per_class=3,
    weights=[0.80, 0.20],
    flip_y=0.03,
    random_state=42
)

# Add noise
X_raw += np.random.RandomState(42).normal(
    0,
    0.5,
    X_raw.shape
)

print(f"Dataset Shape: {X_raw.shape}")

# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_pool, X_test, y_pool, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_pool,
    y_pool,
    test_size=0.20,
    stratify=y_pool,
    random_state=42
)

print("Train Shape :", X_train.shape)
print("Val Shape   :", X_val.shape)
print("Test Shape  :", X_test.shape)


LOADING DATA
Dataset Shape: (11500, 178)
Train Shape : (7360, 178)
Val Shape   : (1840, 178)
Test Shape  : (2300, 178)


In [59]:
# ============================================================
# BLOCK 3 — PREPROCESSING PIPELINES
# ============================================================

# ------------------------------------------------------------
# PIPELINE A
# MinMax → SelectKBest → StandardScaler
# ------------------------------------------------------------

pipe_A = Pipeline([
    ('minmax', MinMaxScaler()),
    ('kbest', SelectKBest(f_classif, k=60)),
    ('standard', StandardScaler())
])

X_tr_A   = pipe_A.fit_transform(X_train, y_train)
X_val_A  = pipe_A.transform(X_val)
X_test_A = pipe_A.transform(X_test)

# ------------------------------------------------------------
# PIPELINE B
# StandardScaler → PCA → MinMax
# ------------------------------------------------------------

pipe_B = Pipeline([
    ('standard', StandardScaler()),
    ('pca', PCA(
        n_components=0.95,
        random_state=42
    )),
    ('minmax', MinMaxScaler())
])

X_tr_B   = pipe_B.fit_transform(X_train, y_train)
X_val_B  = pipe_B.transform(X_val)
X_test_B = pipe_B.transform(X_test)

n_pca = pipe_B.named_steps['pca'].n_components_

print(f"Pipeline A Features : {X_tr_A.shape[1]}")
print(f"Pipeline B PCA Comp : {n_pca}")

Pipeline A Features : 60
Pipeline B PCA Comp : 135


In [60]:
# ============================================================
# BLOCK 4 — BASELINE LOGISTIC REGRESSION
# ============================================================

lr_A = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)

lr_B = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)

# ============================================================
# TRAIN MODELS
# ============================================================

lr_A.fit(X_tr_A, y_train)
lr_B.fit(X_tr_B, y_train)

print("Models Trained Successfully")

Models Trained Successfully


In [61]:
# ============================================================
# BLOCK 5 — METRICS FUNCTION
# ============================================================

def compute_metrics(model, X, y, label=""):

    yp = model.predict(X)

    prob = model.predict_proba(X)[:, 1]

    prec_c, rec_c, _ = precision_recall_curve(y, prob)

    fpr, tpr, _ = roc_curve(y, prob)

    metrics = {

        'acc' : accuracy_score(y, yp),

        'prec' : precision_score(
            y,
            yp,
            zero_division=0
        ),

        'rec' : recall_score(
            y,
            yp,
            zero_division=0
        ),

        'f1' : f1_score(
            y,
            yp,
            zero_division=0
        ),

        'pr_auc' : auc(rec_c, prec_c),

        'roc_auc' : roc_auc_score(y, prob),

        'yp' : yp,
        'prob' : prob,

        'prec_curve' : prec_c,
        'rec_curve' : rec_c,

        'fpr' : fpr,
        'tpr' : tpr
    }

    if label:

        print(f"\n[{label}]")

        print(f"Accuracy : {metrics['acc']*100:.2f}%")

        print(f"Precision : {metrics['prec']*100:.2f}%")

        print(f"Recall : {metrics['rec']*100:.2f}%")

        print(f"F1-Score : {metrics['f1']*100:.2f}%")

        print(f"PR-AUC : {metrics['pr_auc']*100:.2f}%")

        print(f"ROC-AUC : {metrics['roc_auc']*100:.2f}%")

    return metrics

In [62]:
## UNDERFTTING VS OVERFITTING

In [63]:
# ============================================================
# BLOCK 6 — MODEL EVALUATION
# ============================================================

print("\n--- Pipeline A Results ---")

m_A_val = compute_metrics(
    lr_A,
    X_val_A,
    y_val,
    "Pipeline A Validation"
)

m_A_test = compute_metrics(
    lr_A,
    X_test_A,
    y_test,
    "Pipeline A Test"
)

print("\n--- Pipeline B Results ---")

m_B_val = compute_metrics(
    lr_B,
    X_val_B,
    y_val,
    "Pipeline B Validation"
)

m_B_test = compute_metrics(
    lr_B,
    X_test_B,
    y_test,
    "Pipeline B Test"
)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report")

print(classification_report(
    y_test,
    m_A_test['yp'],
    target_names=[
        'Non-Seizure',
        'Seizure'
    ]
))


--- Pipeline A Results ---

[Pipeline A Validation]
Accuracy : 75.60%
Precision : 45.10%
Recall : 77.86%
F1-Score : 57.12%
PR-AUC : 59.67%
ROC-AUC : 83.25%

[Pipeline A Test]
Accuracy : 75.61%
Precision : 44.94%
Recall : 75.00%
F1-Score : 56.21%
PR-AUC : 59.52%
ROC-AUC : 82.90%

--- Pipeline B Results ---

[Pipeline B Validation]
Accuracy : 74.29%
Precision : 43.21%
Recall : 73.70%
F1-Score : 54.48%
PR-AUC : 54.26%
ROC-AUC : 80.43%

[Pipeline B Test]
Accuracy : 73.65%
Precision : 42.39%
Recall : 73.12%
F1-Score : 53.67%
PR-AUC : 56.09%
ROC-AUC : 80.55%

Classification Report
              precision    recall  f1-score   support

 Non-Seizure       0.92      0.76      0.83      1820
     Seizure       0.45      0.75      0.56       480

    accuracy                           0.76      2300
   macro avg       0.68      0.75      0.70      2300
weighted avg       0.82      0.76      0.77      2300



In [64]:
# =============================================================
# SECTION 4 — OVERFITTING & UNDERFITTING
# =============================================================

print("\n" + "="*60)
print(" SECTION 4 — OVERFITTING & UNDERFITTING ")
print("="*60)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import (
    validation_curve,
    learning_curve,
    StratifiedKFold
)

# =============================================================
# 4.1 SCENARIOS
# =============================================================

# -------------------------------------------------------------
# UNDERFITTING
# -------------------------------------------------------------
#
# A) Very Strong Regularization
#    -> C very small
#
# B) Limited Features
#    -> only 2 features
#
# -------------------------------------------------------------
# OVERFITTING
# -------------------------------------------------------------
#
# C) No Regularization
#    -> C very large
#
# D) High-Dimensional Features
#    -> raw 178 features
#

scenarios = [

    # UNDERFITTING A
    {
        "name": "Underfit A (Strong Regularization)",
        "model": LogisticRegression(
            C=0.0001,
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ),
        "X_train": X_tr_A,
        "X_val": X_val_A,
        "color": "purple"
    },

    # UNDERFITTING B
    {
        "name": "Underfit B (Only 2 Features)",
        "model": LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ),
        "X_train": X_tr_A[:, :2],
        "X_val": X_val_A[:, :2],
        "color": "teal"
    },

    # OPTIMAL
    {
        "name": "Optimal Model",
        "model": LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ),
        "X_train": X_tr_A,
        "X_val": X_val_A,
        "color": "green"
    },

    # OVERFITTING A
    {
        "name": "Overfit A (No Regularization)",
        "model": LogisticRegression(
            C=100000,
            max_iter=3000,
            class_weight='balanced',
            random_state=42
        ),
        "X_train": X_tr_A,
        "X_val": X_val_A,
        "color": "orange"
    },

    # OVERFITTING B
    {
        "name": "Overfit B (Raw 178 Features)",
        "model": LogisticRegression(
            C=100000,
            max_iter=3000,
            class_weight='balanced',
            random_state=42
        ),
        "X_train": X_train,
        "X_val": X_val,
        "color": "red"
    }
]

# =============================================================
# 4.2 TRAIN + VALIDATION PERFORMANCE
# =============================================================

print("\nTRAIN vs VALIDATION RESULTS\n")

results = []

for sc in scenarios:

    model = sc["model"]

    model.fit(sc["X_train"], y_train)

    # predictions
    train_pred = model.predict(sc["X_train"])
    val_pred = model.predict(sc["X_val"])

    # metrics
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)

    train_f1 = f1_score(y_train, train_pred)
    val_f1 = f1_score(y_val, val_pred)

    gap = train_acc - val_acc

    results.append({
        "Scenario": sc["name"],
        "Train Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Train F1": train_f1,
        "Validation F1": val_f1,
        "Gap": gap
    })

    print(f"{sc['name']}")
    print(f" Train Accuracy      : {train_acc:.4f}")
    print(f" Validation Accuracy : {val_acc:.4f}")
    print(f" Train F1 Score      : {train_f1:.4f}")
    print(f" Validation F1 Score : {val_f1:.4f}")
    print(f" Generalization Gap  : {gap:.4f}")
    print("-"*50)

# =============================================================
# 4.3 BAR CHART — TRAIN vs VALIDATION
# =============================================================

import matplotlib.pyplot as plt
import numpy as np

labels = [r["Scenario"] for r in results]

train_scores = [r["Train Accuracy"]*100 for r in results]
val_scores = [r["Validation Accuracy"]*100 for r in results]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(14,6))

plt.bar(
    x - width/2,
    train_scores,
    width,
    label='Train Accuracy'
)

plt.bar(
    x + width/2,
    val_scores,
    width,
    label='Validation Accuracy'
)

plt.xticks(x, labels, rotation=10)

plt.ylabel("Accuracy (%)")

plt.title(
    "Overfitting vs Underfitting\n"
    "Train Accuracy vs Validation Accuracy"
)

plt.legend()

plt.grid(alpha=0.3)
plt.savefig('/content/S4_overfit_underfit_bar.png', dpi=160, bbox_inches='tight')
plt.show()

# =============================================================
# 4.4 VALIDATION CURVE
# =============================================================

print("\nGenerating Validation Curve...\n")

C_range = np.logspace(-5, 5, 20)

train_scores, val_scores = validation_curve(

    LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ),

    X_tr_A,
    y_train,

    param_name="C",
    param_range=C_range,

    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),

    scoring="f1",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

# =============================================================
# VALIDATION CURVE PLOT
# =============================================================

plt.figure(figsize=(10,6))

plt.semilogx(
    C_range,
    train_mean,
    marker='o',
    label='Training F1 Score'
)

plt.semilogx(
    C_range,
    val_mean,
    marker='s',
    label='Validation F1 Score'
)

plt.xlabel("C Value (Regularization Strength)")

plt.ylabel("F1 Score")

plt.title(
    "Validation Curve\n"
    "Effect of Regularization on Overfitting/Underfitting"
)

plt.legend()

plt.grid(alpha=0.3)
plt.savefig('/content/S4_validation_curve.png', dpi=160, bbox_inches='tight')
plt.show()

# =============================================================
# 4.5 LEARNING CURVE
# =============================================================

print("\nGenerating Learning Curve...\n")

model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

train_sizes, train_scores, val_scores = learning_curve(

    model,

    X_tr_A,
    y_train,

    cv=5,

    scoring='f1',

    train_sizes=np.linspace(0.1, 1.0, 10),

    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

# =============================================================
# LEARNING CURVE PLOT
# =============================================================

plt.figure(figsize=(10,6))

plt.plot(
    train_sizes,
    train_mean,
    marker='o',
    label='Training F1 Score'
)

plt.plot(
    train_sizes,
    val_mean,
    marker='s',
    label='Validation F1 Score'
)

plt.xlabel("Training Set Size")

plt.ylabel("F1 Score")

plt.title(
    "Learning Curve\n"
    "Training Size vs Model Performance"
)

plt.legend()

plt.grid(alpha=0.3)
plt.savefig('/content/S4_learning_curve.png', dpi=160, bbox_inches='tight')
plt.show()


 SECTION 4 — OVERFITTING & UNDERFITTING 

TRAIN vs VALIDATION RESULTS

Underfit A (Strong Regularization)
 Train Accuracy      : 0.7450
 Validation Accuracy : 0.7408
 Train F1 Score      : 0.5467
 Validation F1 Score : 0.5470
 Generalization Gap  : 0.0042
--------------------------------------------------
Underfit B (Only 2 Features)
 Train Accuracy      : 0.5793
 Validation Accuracy : 0.5668
 Train F1 Score      : 0.3574
 Validation F1 Score : 0.3308
 Generalization Gap  : 0.0125
--------------------------------------------------
Optimal Model
 Train Accuracy      : 0.7678
 Validation Accuracy : 0.7560
 Train F1 Score      : 0.5779
 Validation F1 Score : 0.5712
 Generalization Gap  : 0.0118
--------------------------------------------------
Overfit A (No Regularization)
 Train Accuracy      : 0.7694
 Validation Accuracy : 0.7533
 Train F1 Score      : 0.5798
 Validation F1 Score : 0.5668
 Generalization Gap  : 0.0162
--------------------------------------------------
Overfit B (Raw 1

In [65]:
# =============================================================
# SECTION 5 — REGULARIZATION STUDY
# =============================================================

print("\n" + "="*60)
print(" SECTION 5 — REGULARIZATION STUDY ")
print("="*60)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =============================================================
# 5.1 DEFINE REGULARIZATION MODELS
# =============================================================

models = {

    "L1 (Lasso)": LogisticRegression(
        penalty='l1',
        solver='saga',
        C=1.0,
        max_iter=4000,
        class_weight='balanced',
        random_state=42
    ),

    "L2 (Ridge)": LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        C=1.0,
        max_iter=4000,
        class_weight='balanced',
        random_state=42
    ),

    "Elastic Net": LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        l1_ratio=0.5,
        C=1.0,
        max_iter=4000,
        class_weight='balanced',
        random_state=42
    )
}

# =============================================================
# 5.2 TRAIN + EVALUATE MODELS
# =============================================================

results = []

print("\nMODEL PERFORMANCE\n")

for name, model in models.items():

    # train
    model.fit(X_tr_A, y_train)

    # predictions
    y_pred_train = model.predict(X_tr_A)
    y_pred_val = model.predict(X_val_A)

    # metrics
    train_acc = accuracy_score(y_train, y_pred_train)
    val_acc = accuracy_score(y_val, y_pred_val)

    train_f1 = f1_score(y_train, y_pred_train)
    val_f1 = f1_score(y_val, y_pred_val)

    # coefficients
    coef = model.coef_[0]

    # sparsity calculation
    zero_coef = np.sum(coef == 0)

    sparsity = (zero_coef / len(coef)) * 100

    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Train F1": train_f1,
        "Validation F1": val_f1,
        "Zero Coefficients": zero_coef,
        "Sparsity (%)": sparsity
    })

    print(f"{name}")
    print(f" Train Accuracy      : {train_acc:.4f}")
    print(f" Validation Accuracy : {val_acc:.4f}")
    print(f" Train F1 Score      : {train_f1:.4f}")
    print(f" Validation F1 Score : {val_f1:.4f}")
    print(f" Zero Coefficients   : {zero_coef}")
    print(f" Sparsity            : {sparsity:.2f}%")
    print("-"*50)

# =============================================================
# RESULTS TABLE
# =============================================================

results_df = pd.DataFrame(results)

print("\nSUMMARY TABLE\n")

display(results_df)

# =============================================================
# 5.3 SPARSITY VISUALIZATION
# =============================================================

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Sparsity (%)"]
)

plt.ylabel("Sparsity (%)")

plt.title(
    "Feature Selection Effect of Regularization\n"
    "(Percentage of Zero Coefficients)"
)

plt.grid(alpha=0.3)
plt.savefig('/content/S5_sparsity_bar.png', dpi=160, bbox_inches='tight')
plt.show()

# =============================================================
# 5.4 COEFFICIENT DISTRIBUTION
# =============================================================

plt.figure(figsize=(12,6))

for name, model in models.items():

    coef = model.coef_[0]

    plt.hist(
        coef,
        bins=30,
        alpha=0.5,
        label=name
    )

plt.xlabel("Coefficient Value")

plt.ylabel("Frequency")

plt.title(
    "Distribution of Logistic Regression Coefficients"
)

plt.legend()

plt.grid(alpha=0.3)
plt.savefig('/content/S5_coef_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

# =============================================================
# 5.5 NUMBER OF NON-ZERO FEATURES
# =============================================================

nonzero_features = []

for name, model in models.items():

    coef = model.coef_[0]

    nonzero = np.sum(coef != 0)

    nonzero_features.append(nonzero)

plt.figure(figsize=(8,5))

plt.bar(
    list(models.keys()),
    nonzero_features
)

plt.ylabel("Number of Non-Zero Features")

plt.title(
    "Remaining Features After Regularization"
)

plt.grid(alpha=0.3)
plt.savefig('/content/S5_nonzero_features.png', dpi=160, bbox_inches='tight')
plt.show()


 SECTION 5 — REGULARIZATION STUDY 

MODEL PERFORMANCE

L1 (Lasso)
 Train Accuracy      : 0.7689
 Validation Accuracy : 0.7543
 Train F1 Score      : 0.5795
 Validation F1 Score : 0.5679
 Zero Coefficients   : 3
 Sparsity            : 5.00%
--------------------------------------------------
L2 (Ridge)
 Train Accuracy      : 0.7678
 Validation Accuracy : 0.7560
 Train F1 Score      : 0.5779
 Validation F1 Score : 0.5712
 Zero Coefficients   : 0
 Sparsity            : 0.00%
--------------------------------------------------
Elastic Net
 Train Accuracy      : 0.7692
 Validation Accuracy : 0.7560
 Train F1 Score      : 0.5796
 Validation F1 Score : 0.5712
 Zero Coefficients   : 1
 Sparsity            : 1.67%
--------------------------------------------------

SUMMARY TABLE



,Model,Train Accuracy,Validation Accuracy,Train F1,Validation F1,Zero Coefficients,Sparsity (%)
0,L1 (Lasso),0.768886,0.754348,0.579481,0.567878,3,5.000000
1,L2 (Ridge),0.767799,0.755978,0.577920,0.571156,0,0.000000
2,Elastic Net,0.769158,0.755978,0.579560,0.571156,1,1.666667


In [66]:
!pip install imbalanced-learn

In [67]:
# ═══════════════════════════════════════════════════════════════
#  SECTION 6 — HANDLING CLASS IMBALANCE
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  SECTION 6 — HANDLING CLASS IMBALANCE")
print("="*60)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# --------------------------------------------------------------
# Helper function
# --------------------------------------------------------------
def evaluate_model(name, model, X_tr, y_tr, X_val, y_val):

    model.fit(X_tr, y_tr)

    yp   = model.predict(X_val)
    prob = model.predict_proba(X_val)[:, 1]

    precision = precision_score(y_val, yp)
    recall    = recall_score(y_val, yp)
    f1        = f1_score(y_val, yp)
    pr_auc    = roc_auc_score(y_val, prob)

    return {
        'Method'    : name,
        'Precision' : precision * 100,
        'Recall'    : recall * 100,
        'F1'        : f1 * 100,
        'ROC-AUC'   : pr_auc * 100
    }

# --------------------------------------------------------------
# BASELINE (No balancing)
# --------------------------------------------------------------
lr_base = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

res_base = evaluate_model(
    "Baseline",
    lr_base,
    X_tr_A, y_train,
    X_val_A, y_val
)

# --------------------------------------------------------------
# CLASS WEIGHTING
# --------------------------------------------------------------
lr_weighted = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

res_weight = evaluate_model(
    "Class Weighting",
    lr_weighted,
    X_tr_A, y_train,
    X_val_A, y_val
)

# --------------------------------------------------------------
# SMOTE OVERSAMPLING
# --------------------------------------------------------------
smote = SMOTE(random_state=42)

X_smote, y_smote = smote.fit_resample(X_tr_A, y_train)

print("\nSMOTE Resampled Shape:")
print("Before:", X_tr_A.shape, np.bincount(y_train))
print("After :", X_smote.shape, np.bincount(y_smote))

lr_smote = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

res_smote = evaluate_model(
    "SMOTE Oversampling",
    lr_smote,
    X_smote, y_smote,
    X_val_A, y_val
)

# --------------------------------------------------------------
# RANDOM UNDERSAMPLING
# --------------------------------------------------------------
under = RandomUnderSampler(random_state=42)

X_under, y_under = under.fit_resample(X_tr_A, y_train)

print("\nUndersampling Shape:")
print("Before:", X_tr_A.shape, np.bincount(y_train))
print("After :", X_under.shape, np.bincount(y_under))

lr_under = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

res_under = evaluate_model(
    "Undersampling",
    lr_under,
    X_under, y_under,
    X_val_A, y_val
)

# --------------------------------------------------------------
# CREATE RESULTS TABLE
# --------------------------------------------------------------
results_df = pd.DataFrame([
    res_base,
    res_weight,
    res_smote,
    res_under
])

print("\n")
print(results_df.round(2))


  SECTION 6 — HANDLING CLASS IMBALANCE

SMOTE Resampled Shape:
Before: (7360, 60) [5822 1538]
After : (11644, 60) [5822 5822]

Undersampling Shape:
Before: (7360, 60) [5822 1538]
After : (3076, 60) [1538 1538]


               Method  Precision  Recall     F1  ROC-AUC
0            Baseline      71.93   42.71  53.59    83.12
1     Class Weighting      45.10   77.86  57.12    83.25
2  SMOTE Oversampling      45.55   77.34  57.34    83.35
3       Undersampling      45.98   78.91  58.10    83.38


In [68]:
# ──────────────────────────────────────────────────────────────
# Figure S6-1 — Precision vs Recall Tradeoff
# ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Section 6 — Handling Class Imbalance",
    fontsize=15,
    fontweight='bold'
)

methods = results_df['Method']

# --------------------------------------------------------------
# Precision vs Recall
# --------------------------------------------------------------
ax = axes[0]

x = np.arange(len(methods))
w = 0.35

ax.bar(x - w/2,
       results_df['Precision'],
       width=w,
       color='#457B9D',
       label='Precision')

ax.bar(x + w/2,
       results_df['Recall'],
       width=w,
       color='#E63946',
       label='Recall')

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=10)

ax.set_ylabel('Score (%)')
ax.set_title('Precision vs Recall')

ax.legend()
ax.grid(alpha=0.3)

# --------------------------------------------------------------
# F1 + ROC-AUC
# --------------------------------------------------------------
ax = axes[1]

ax.plot(methods,
        results_df['F1'],
        marker='o',
        linewidth=2.5,
        label='F1-Score')

ax.plot(methods,
        results_df['ROC-AUC'],
        marker='s',
        linewidth=2.5,
        label='ROC-AUC')

ax.set_ylabel('Score (%)')
ax.set_title('F1 and ROC-AUC Comparison')

ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

out = '/content/S6_class_imbalance.png'

plt.savefig(out, dpi=160, bbox_inches='tight')

plt.show()

print(f"\n[Saved] {out}")


[Saved] /content/S6_class_imbalance.png
